In [71]:
import pandas as pd
import numpy as np
import psycopg2
import json
import csv
import os

In [ ]:
with open('./dataset/archive/data/mpd.slice.0-999.json', 'r') as file:
    json_data = json.load(file)

In [9]:
json_data['playlists'][0]
test =json_data['playlists'][0]['tracks'][0]
#print(test['track_name'], test['duration_ms'])
json_data['playlists'][0]


{'name': 'Throwbacks',
 'collaborative': 'false',
 'pid': 0,
 'modified_at': 1493424000,
 'num_tracks': 52,
 'num_albums': 47,
 'num_followers': 1,
 'tracks': [{'pos': 0,
   'artist_name': 'Missy Elliott',
   'track_uri': 'spotify:track:0UaMYEvWZi0ZqiDOoHU3YI',
   'artist_uri': 'spotify:artist:2wIVse2owClT7go1WT98tk',
   'track_name': 'Lose Control (feat. Ciara & Fat Man Scoop)',
   'album_uri': 'spotify:album:6vV5UrXcfyQD1wu4Qo2I9K',
   'duration_ms': 226863,
   'album_name': 'The Cookbook'},
  {'pos': 1,
   'artist_name': 'Britney Spears',
   'track_uri': 'spotify:track:6I9VzXrHxO9rA9A5euc8Ak',
   'artist_uri': 'spotify:artist:26dSoYclwsYLMAKD3tpOr4',
   'track_name': 'Toxic',
   'album_uri': 'spotify:album:0z7pVBGOD7HCIB7S8eLkLI',
   'duration_ms': 198800,
   'album_name': 'In The Zone'},
  {'pos': 2,
   'artist_name': 'Beyoncé',
   'track_uri': 'spotify:track:0WqIKmW4BTrj3eJFmnCKMv',
   'artist_uri': 'spotify:artist:6vWDO969PvNqNYHIOW5v0m',
   'track_name': 'Crazy In Love',
   'alb

In [ ]:
with open('./dataset/archive/data/mpd.slice.0-999.json', 'r') as file:
    json_data = json.load(file)
df_csv = pd.read_csv('./dataset/spotify_clean_data.csv')

In [8]:
playlist = json_data['playlists']
playlist[1]

{'name': 'Awesome Playlist',
 'collaborative': 'false',
 'pid': 1,
 'modified_at': 1506556800,
 'num_tracks': 39,
 'num_albums': 23,
 'num_followers': 1,
 'tracks': [{'pos': 0,
   'artist_name': 'Survivor',
   'track_uri': 'spotify:track:2HHtWyy5CgaQbC7XSoOb0e',
   'artist_uri': 'spotify:artist:26bcq2nyj5GB7uRr558iQg',
   'track_name': 'Eye of the Tiger',
   'album_uri': 'spotify:album:4PT9VulQaQP6XR1xBI2x1W',
   'duration_ms': 243773,
   'album_name': 'Eye Of The Tiger'},
  {'pos': 1,
   'artist_name': 'Daniel Tidwell',
   'track_uri': 'spotify:track:1MYYt7h6amcrauCOoso3Gx',
   'artist_uri': 'spotify:artist:7zdmbPudNX4SQJXnYIuCTC',
   'track_name': 'Libera Me From Hell (Tengen Toppa Gurren Lagann)',
   'album_uri': 'spotify:album:3q8vR3PFV8kG1m1Iv8DpKq',
   'duration_ms': 70294,
   'album_name': 'Versus Hollywood'},
  {'pos': 2,
   'artist_name': 'Daniel Tidwell',
   'track_uri': 'spotify:track:3x2mJ2bjCIU70NrH49CtYR',
   'artist_uri': 'spotify:artist:7zdmbPudNX4SQJXnYIuCTC',
   'trac

In [77]:
keep_columns = ['track_id', 'duration_ms_dataset']

person = {}
person_num = -1
max_files = 0

df_csv['key_name'] = df_csv['track_name'].astype(str).str.lower().str.strip()

directory = './dataset/archive/data'
for filename in os.listdir(directory):
    if max_files >= 50:
        break
    max_files+=1

    if filename.endswith('.json'):
        complete_path = os.path.join(directory, filename)
        try:
            with open(complete_path, 'r') as file:
                json_data = json.load(file)
            playlist = json_data['playlists']
        except Exception as e:
            print(f"Error reading {filename}: {e}")
            continue

    limit = min(1000, len(playlist))
    for i in range(limit):
        if i % 5 == 0:
            person_num += 1
            person[person_num] = pd.DataFrame()

        df_playlist = pd.DataFrame(playlist[i]['tracks']).dropna()
        df_playlist['key_name'] = df_playlist['track_name'].astype(str).str.lower().str.strip()

        df_final = pd.merge(
            df_playlist,          # Tabela da Esquerda (Sua Playlist)
            df_csv,               # Tabela da Direita (O Dataset Gigante)
            on='key_name',        # A chave normalizada que criamos
            how='inner',          # Inner = Só o que tem match
            suffixes=('_playlist', '_dataset') # Caso tenham colunas com mesmo nome (ex: duration_ms)
        )

        #df_final = df_final.drop_duplicates(subset=['artist_name', 'track_uri'], keep='first')
        df_final = df_final[keep_columns].copy()
        df_final['username'] = person_num + 1

        if person[person_num].empty:
            person[person_num] = df_final
        else:
            person[person_num] = pd.concat([person[person_num], df_final])


In [78]:
for i in range(len(person)):
    if i == 0:
        person[i].to_csv('./dataset/fds3.csv', mode='a', index=False)
    else:
        person[i].to_csv('./dataset/fds3.csv', mode='a', index=False, header=False)

In [74]:
len(person)

2000